**Soldaki menüden "files" kısmına girip google drive'ı mount ederek aşağıdaki celleri çalıştırmaya başlayacağız.**

**Runtime'ı gpu seçtiğinden emin ol.**

SMPL_JOINT_NAMES = [ "pelvis", "left_hip", "right_hip", "spine1", "left_knee", "right_knee", "spine2", "left_ankle", "right_ankle", "spine3", "left_foot", "right_foot", "neck", "left_collar", "right_collar", "head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow", "left_wrist", "right_wrist", "left_hand", "right_hand", ]

In [1]:
import torch
import torchvision
import numpy as np
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

MODEL_PATH = "/content/drive/MyDrive/torchscript/nlf_l_multi_0.3.2.torchscript"

model = torch.jit.load(MODEL_PATH).to(device).eval()

print("Model loaded.")

Using device: cuda
Model loaded.


In [12]:
def to_jsonable(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, dict):
        return {k: to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_jsonable(v) for v in x]
    return x


def run_nlf_on_frame(frame_bgr):
    # BGR → RGB
    frame_rgb = frame_bgr[:, :, ::-1]
    img = Image.fromarray(frame_rgb).convert("RGB")

    img_t = torchvision.transforms.functional.to_tensor(img).to(device)
    frame_batch = img_t.unsqueeze(0)

    with torch.inference_mode():
        pred = model.detect_smpl_batched(frame_batch, model_name='smpl')

    pred = to_jsonable(pred)

    joints_3d = pred.get("joints3d", None)
    joints_2d = pred.get("joints2d", None)

    return joints_3d, joints_2d

**Tek video ise bu cell'i çalıştır pathi düzenleyip.**

In [11]:
import cv2
import json
from tqdm import tqdm

VIDEO_PATH = "/content/drive/MyDrive/runalyst_test_videos_edited_28feb/heel_strike_alper.mov"
OUTPUT_PATH = "/content/drive/MyDrive/runalyst_test_videos_edited_28feb/nlf_outputs/heel_strike_alper2.jsonl"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Video açılamadı.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Total frames:", total_frames)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:

    frame_index = 0

    with torch.inference_mode():
        for _ in tqdm(range(total_frames)):
            ok, frame = cap.read()
            if not ok:
                break

            timestamp_sec = frame_index / fps if fps > 0 else None

            try:
                joints_3d, joints_2d = run_nlf_on_frame(frame)

                record = {
                    "frame_index": frame_index,
                    "timestamp_sec": timestamp_sec,
                    "joints_3d": joints_3d,
                    "joints_2d": joints_2d,
                }

            except Exception as e:
                record = {
                    "frame_index": frame_index,
                    "timestamp_sec": timestamp_sec,
                    "error": str(e)
                }

            # Tek satır JSON
            f.write(json.dumps(record) + "\n")

            frame_index += 1

cap.release()
print("Bitti. Kaydedildi:", OUTPUT_PATH)

FPS: 56.97151424287856
Total frames: 190


 96%|█████████▌| 182/190 [01:24<00:03,  2.15it/s]

Bitti. Kaydedildi: /content/drive/MyDrive/runalyst_test_videos_edited_28feb/nlf_outputs/heel_strike_alper2.jsonl


**Video folderı verecksen bu cell'i çalıştır.** hem 3d hem 2d joint kaydeder.

In [ ]:
import os
import cv2
import json
from tqdm import tqdm

# BURAYA VİDEO KLASÖRÜNÜ YAZ
VIDEO_FOLDER = "/content/drive/MyDrive/runalyst_test_videos_edited_28feb"
NEW_OUTPUT_FOLDER = os.path.join(VIDEO_FOLDER, "nlf_outputs")
os.makedirs(NEW_OUTPUT_FOLDER, exist_ok=True)
# Desteklenen uzantılar
VIDEO_EXTENSIONS = (".mp4", ".mov", ".avi", ".mkv")

video_files = [
    f for f in os.listdir(VIDEO_FOLDER)
    if f.lower().endswith(VIDEO_EXTENSIONS)
]

print(f"Bulunan video sayısı: {len(video_files)}")

for video_name in video_files:

    video_path = os.path.join(VIDEO_FOLDER, video_name)
    output_name = f"output_{os.path.splitext(video_name)[0]}.jsonl"
    output_path = os.path.join(NEW_OUTPUT_FOLDER, output_name)

    print("\n====================================")
    print("Processing:", video_name)
    print("Saving to :", output_name)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Video açılamadı:", video_name)
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)

    with open(output_path, "w", encoding="utf-8") as f:

        frame_index = 0

        while True:
            ok, frame = cap.read()
            if not ok:
                print("Video stream ended at frame:", frame_index)
                break

            timestamp_sec = frame_index / fps if fps > 0 else None

            try:
                joints_3d, joints_2d = run_nlf_on_frame(frame)

                record = {
                    "frame_index": frame_index,
                    "timestamp_sec": timestamp_sec,
                    "joints_3d": joints_3d,
                    "joints_2d": joints_2d,
                }

            except Exception as e:
                record = {
                    "frame_index": frame_index,
                    "timestamp_sec": timestamp_sec,
                    "error": str(e)
                }

            f.write(json.dumps(record) + "\n")

            frame_index += 1

    cap.release()
    print("Finished:", video_name)

print("\n✅ All videos processed.")

Bulunan video sayısı: 32

Processing: belden_kambur_baris.mov
Saving to : output_belden_kambur_baris.jsonl


Prod için sadece 3d jointleri kaydeden versiyon:

In [ ]:
import os
import cv2
import json
from tqdm import tqdm

# BURAYA VİDEO KLASÖRÜNÜ YAZ
VIDEO_FOLDER = "/content/drive/MyDrive/runalyst_test_videos_edited_28feb"
NEW_OUTPUT_FOLDER = os.path.join(VIDEO_FOLDER, "nlf_outputs")
os.makedirs(NEW_OUTPUT_FOLDER, exist_ok=True)

VIDEO_EXTENSIONS = (".mp4", ".mov", ".avi", ".mkv")

video_files = [
    f for f in os.listdir(VIDEO_FOLDER)
    if f.lower().endswith(VIDEO_EXTENSIONS)
]

print(f"Bulunan video sayısı: {len(video_files)}")

for video_name in video_files:

    video_path = os.path.join(VIDEO_FOLDER, video_name)
    output_name = f"output_{os.path.splitext(video_name)[0]}.jsonl"
    output_path = os.path.join(NEW_OUTPUT_FOLDER, output_name)

    print("\n====================================")
    print("Processing:", video_name)
    print("Saving to :", output_name)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Video açılamadı:", video_name)
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)

    with open(output_path, "w", encoding="utf-8") as f:

        frame_index = 0

        while True:
            ok, frame = cap.read()
            if not ok:
                print("Video stream ended at frame:", frame_index)
                break

            timestamp_sec = frame_index / fps if fps > 0 else None

            try:
                # SADECE 3D
                joints_3d, _ = run_nlf_on_frame(frame)

                record = {
                    "frame_index": frame_index,
                    "timestamp_sec": timestamp_sec,
                    "joints_3d": joints_3d,
                }

            except Exception as e:
                record = {
                    "frame_index": frame_index,
                    "timestamp_sec": timestamp_sec,
                    "error": str(e)
                }

            f.write(json.dumps(record) + "\n")

            frame_index += 1

    cap.release()
    print("Finished:", video_name)

print("\n✅ All videos processed.")